# File I/O Benchmark Development and Final Experiment

This notebook documents the design, implementation, execution and quality assurance of the file I/O benchmark used to compare Node.js, Bun and Deno.

The benchmark evaluates two file operations:

1. sequential whole-file reading; and
2. sequential whole-file writing.

Statistical analysis and interpretation are deferred until data collection has been completed for all dependent variables.

## 1. Purpose of the File I/O Benchmark

The file I/O benchmark measures how efficiently Node.js, Bun and Deno read and write a fixed binary file.

Two dependent measures are recorded:

- operation duration in milliseconds; and
- effective transfer throughput in mebibytes per second.

The benchmark measures application-level file I/O through each runtime's built-in file-system API. It is not intended to measure the maximum theoretical performance of the storage hardware independently of the runtime.

## 2. File I/O Workloads

A deterministic binary fixture of 100 MiB was used for both operations.

### Sequential read workload

Each runtime:

1. read the complete fixture once outside the measured interval to establish a consistent warm-cache condition;
2. started a high-resolution internal timer;
3. read the complete fixture into memory;
4. stopped the timer; and
5. validated the number and boundary values of the bytes read.

### Sequential write workload

Each runtime:

1. loaded the deterministic fixture into memory outside the measured interval;
2. started a high-resolution internal timer;
3. wrote the complete 100 MiB payload to a new output file;
4. stopped the timer; and
5. returned the recorded duration and byte count.

The output file was validated and removed by the external experiment runner outside the measured interval.

## 3. Measurement Scope

The read workload represents a buffered warm-cache whole-file read. An unrecorded read was performed immediately before the measured read to reduce differences caused by the operating system's file cache.

The experiment did not attempt to clear the Windows file-system cache because a reliable and runtime-neutral cache-clearing mechanism was not available. The results should therefore not be interpreted as uncached physical-disk read speeds.

The write workload measures the time taken for the runtime's buffered whole-file write operation to complete. It does not explicitly force a hardware-level disk flush. Consequently, the metric represents application-observed buffered write completion rather than guaranteed physical persistence to the storage medium.

These boundaries were applied consistently to Node.js, Bun and Deno.

## 4. Timing Tool and Justification

File I/O duration was measured inside each JavaScript benchmark program using the runtime's high-resolution `performance.now()` timer.

The timer began immediately before the measured file-system operation and ended immediately after the operation completed.

An internal timer was selected because the intended construct was the duration of the file operation itself. Measuring the complete runtime command using an external command benchmark such as Hyperfine would also include:

- runtime process creation;
- JavaScript runtime initialisation;
- source-file loading;
- fixture preparation;
- output validation; and
- process termination.

Runtime startup was measured separately as the cold-start dependent variable. Including startup overhead again in the file I/O metric would reduce construct validity and make it difficult to determine whether observed differences originated from file I/O or process startup.

Python was used only to automate execution, randomise test order, validate output files and collect results. Python did not produce the measured file I/O duration.

### 2.1 Binary Fixture Format

The file I/O fixture was stored using the `.bin` extension because it contained raw binary bytes rather than human-readable text.

A binary fixture was selected to avoid introducing additional processing that could affect the comparison, including:

- character encoding and decoding;
- newline conversion;
- differences between character counts and byte counts;
- JSON parsing;
- string creation and manipulation; and
- runtime-specific text handling.

Using binary data allowed the benchmark to focus more directly on reading and writing a fixed number of bytes.

The fixture contained a deterministic repeating byte pattern covering values from 0 to 255. This made it possible to verify:

- the exact file size;
- that all runtimes used the same input data;
- that the correct number of bytes was read or written; and
- that selected boundary-byte values matched the expected content.

The `.bin` extension was used as a descriptive indication that the file contained binary data. It was not a technical requirement; extensions such as `.dat` or `.raw` could have been used without changing how the runtimes processed the file.

The final fixture size was 100 MiB, equivalent to 104,857,600 bytes.

In [1]:
library(dplyr)

candidate_manifest_paths <- c(
  "data/raw/file_io/file_io_final_manifest.csv",
  "../data/raw/file_io/file_io_final_manifest.csv"
)

available_manifest_paths <- candidate_manifest_paths[
  file.exists(candidate_manifest_paths)
]

if (length(available_manifest_paths) == 0) {
  stop("The final file I/O manifest could not be found.")
}

file_io_manifest_path <- normalizePath(
  available_manifest_paths[1],
  winslash = "/",
  mustWork = TRUE
)

file_io_final <- read.csv(
  file_io_manifest_path,
  stringsAsFactors = FALSE
)

to_logical <- function(value) {
  if (is.logical(value)) {
    return(value)
  }

  tolower(as.character(value)) %in% c(
    "true",
    "1",
    "yes"
  )
}

file_io_final <- file_io_final |>
  mutate(
    session_id = as.integer(session_id),
    repetition = as.integer(repetition),
    sequence = as.integer(sequence),
    bytes = as.numeric(bytes),
    duration_ms = as.numeric(duration_ms),
    mib_per_second = as.numeric(mib_per_second),
    command_wall_time_ms = as.numeric(
      command_wall_time_ms
    ),
    exit_code = as.integer(exit_code),
    output_file_validated = to_logical(
      output_file_validated
    ),
    output_file_removed = to_logical(
      output_file_removed
    )
  )

cat("Manifest path:\n")
cat(file_io_manifest_path, "\n\n")

cat("Rows:", nrow(file_io_final), "\n")
cat("Columns:", ncol(file_io_final), "\n")


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




Manifest path:
D:/FOLO_PROJECTS/MASTERS/javascript-runtime-performance-study/data/raw/file_io/file_io_final_manifest.csv 

Rows: 60 
Columns: 23 


In [2]:
file_io_group_counts <- file_io_final |>
  count(
    runtime,
    operation,
    name = "observations"
  ) |>
  arrange(
    operation,
    runtime
  )

file_io_group_counts

runtime,operation,observations
<chr>,<chr>,<int>
bun,read,10
deno,read,10
node,read,10
bun,write,10
deno,write,10
node,write,10


In [3]:
file_io_session_counts <- file_io_final |>
  count(
    session_id,
    runtime,
    operation,
    name = "observations"
  ) |>
  arrange(
    session_id,
    operation,
    runtime
  )

file_io_session_counts

session_id,runtime,operation,observations
<int>,<chr>,<chr>,<int>
1,bun,read,5
1,deno,read,5
1,node,read,5
1,bun,write,5
1,deno,write,5
1,node,write,5
2,bun,read,5
2,deno,read,5
2,node,read,5


In [4]:
file_io_duplicate_check <- file_io_final |>
  count(
    runtime,
    operation,
    repetition,
    name = "occurrences"
  ) |>
  filter(
    occurrences != 1
  )

file_io_duplicate_check

runtime,operation,repetition,occurrences
<chr>,<chr>,<int>,<int>


In [5]:
expected_file_io_design <- expand.grid(
  runtime = c(
    "node",
    "bun",
    "deno"
  ),
  operation = c(
    "read",
    "write"
  ),
  repetition = 1:10,
  stringsAsFactors = FALSE
)

missing_file_io_observations <- expected_file_io_design |>
  anti_join(
    file_io_final |>
      select(
        runtime,
        operation,
        repetition
      ),
    by = c(
      "runtime",
      "operation",
      "repetition"
    )
  )

missing_file_io_observations

runtime,operation,repetition
<chr>,<chr>,<int>


In [6]:
file_io_audit_summary <- data.frame(
  check = c(
    "Total observations",
    "Unique run identifiers",
    "Runtime-operation groups",
    "Groups with 10 observations",
    "Session-runtime-operation groups",
    "Session groups with 5 observations",
    "Duplicate configurations",
    "Missing configurations",
    "Failed observations",
    "Incorrect byte counts",
    "Missing durations",
    "Non-positive durations",
    "Missing throughput values",
    "Non-positive throughput values",
    "Non-zero exit codes",
    "Unvalidated output files",
    "Temporary output files not removed"
  ),

  result = c(
    nrow(file_io_final),

    n_distinct(
      file_io_final$run_id
    ),

    nrow(
      file_io_group_counts
    ),

    sum(
      file_io_group_counts$observations == 10
    ),

    nrow(
      file_io_session_counts
    ),

    sum(
      file_io_session_counts$observations == 5
    ),

    nrow(
      file_io_duplicate_check
    ),

    nrow(
      missing_file_io_observations
    ),

    sum(
      file_io_final$status != "success"
    ),

    sum(
      file_io_final$bytes != 104857600,
      na.rm = TRUE
    ),

    sum(
      is.na(file_io_final$duration_ms)
    ),

    sum(
      file_io_final$duration_ms <= 0,
      na.rm = TRUE
    ),

    sum(
      is.na(file_io_final$mib_per_second)
    ),

    sum(
      file_io_final$mib_per_second <= 0,
      na.rm = TRUE
    ),

    sum(
      file_io_final$exit_code != 0,
      na.rm = TRUE
    ),

    sum(
      !file_io_final$output_file_validated
    ),

    sum(
      !file_io_final$output_file_removed
    )
  ),

  expected = c(
    60,
    60,
    6,
    6,
    12,
    12,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
  )
) |>
  mutate(
    passed = result == expected
  )

file_io_audit_summary

check,result,expected,passed
<chr>,<int>,<dbl>,<lgl>
Total observations,60,60,TRUE
Unique run identifiers,60,60,TRUE
Runtime-operation groups,6,6,TRUE
Groups with 10 observations,6,6,TRUE
Session-runtime-operation groups,12,12,TRUE
Session groups with 5 observations,12,12,TRUE
Duplicate configurations,0,0,TRUE
Missing configurations,0,0,TRUE
Failed observations,0,0,TRUE


In [7]:
# save the processed dataset

project_root <- if (
  dir.exists("data")
) {
  "."
} else {
  ".."
}

processed_directory <- file.path(
  project_root,
  "data",
  "processed",
  "file_io"
)

table_directory <- file.path(
  project_root,
  "results",
  "tables",
  "file_io"
)

dir.create(
  processed_directory,
  recursive = TRUE,
  showWarnings = FALSE
)

dir.create(
  table_directory,
  recursive = TRUE,
  showWarnings = FALSE
)

file_io_processed_path <- file.path(
  processed_directory,
  "file_io_final.csv"
)

write.csv(
  file_io_final,
  file_io_processed_path,
  row.names = FALSE
)

write.csv(
  file_io_audit_summary,
  file.path(
    table_directory,
    "file_io_final_audit_summary.csv"
  ),
  row.names = FALSE
)

cat(
  "Processed dataset saved to:\n",
  normalizePath(
    file_io_processed_path,
    winslash = "/",
    mustWork = TRUE
  )
)

Processed dataset saved to:
 D:/FOLO_PROJECTS/MASTERS/javascript-runtime-performance-study/data/processed/file_io/file_io_final.csv

## 5. Final Data Collection and Quality Assurance

The final file I/O experiment produced 60 observations. The experimental design consisted of:

- three JavaScript runtimes;
- two file operations: sequential read and sequential write; and
- ten repetitions per runtime and operation combination.

The observations were divided across two balanced sessions. Each session contained five repetitions of all six runtime-operation combinations, producing 30 observations per session.

The order of the six combinations was randomised independently within each repetition using a fixed and recorded random seed.

A post-collection quality audit confirmed that:

- all 60 planned observations were present;
- all run identifiers were unique;
- each runtime-operation group contained exactly ten observations;
- each session contained five observations for every runtime-operation group;
- no configurations were duplicated;
- no planned configurations were missing;
- all benchmark processes completed successfully;
- every operation processed exactly 104,857,600 bytes;
- all durations and throughput values were present and greater than zero;
- all benchmark processes returned an exit code of zero;
- all written output files passed validation; and
- all temporary output files were removed after validation.

The audited dataset was saved as:

`data/processed/file_io/file_io_final.csv`

The raw JSON observations, manifest, execution plans, runtime logs, fixture hash and benchmark-program hashes were retained.

## 6. Deferred Analysis

Descriptive statistics, visualisations, statistical testing and interpretation of the file I/O results were deferred until data collection and quality assurance had been completed for all dependent variables.